
# Notebook 4: Duplicate Data Handling

## Objective
Understand how to identify, analyze, and handle duplicate records
without accidentally removing valid data.

## Learning Outcomes
- Understand exact and partial duplicates.
- Identify duplicates using duplicated().
- Remove duplicates using drop_duplicates().
- Detect duplicates using selected columns.
- Apply business rules when handling duplicate records.
- Understand the risks of blindly deleting duplicates.


In [1]:
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)
print("Libraries imported successfully!")

Pandas version: 3.0.3
Libraries imported successfully!



## 1. What Are Duplicate Records?

Duplicate records are records that repeat the same information
or represent the same real-world entity or event.

Duplicates may occur because of repeated data entry, system
integration, file merging, or repeated data collection.

Not every repeated value is an error. Two customers may share
the same name, and two transactions may have the same amount.

Duplicates should be investigated before removal.


In [2]:
data = {
    "Customer_ID": [101, 102, 103, 102, 104, 105, 106],
    "Name": [
        "Ravi", "Priya", "Arun", "Priya",
        "Sneha", "Ravi", "Kiran"
    ],
    "City": [
        "Hyderabad", "Chennai", "Mumbai", "Chennai",
        "Delhi", "Hyderabad", "Pune"
    ],
    "Age": [25, 30, 35, 30, 28, 25, 40],
    "Purchase_Amount": [500, 800, 650, 800, 450, 500, 900]
}

df_duplicates = pd.DataFrame(data)

print("Sample dataset:")
display(df_duplicates)

Sample dataset:


,Customer_ID,Name,City,Age,Purchase_Amount
0,101,Ravi,Hyderabad,25,500
1,102,Priya,Chennai,30,800
2,103,Arun,Mumbai,35,650
3,102,Priya,Chennai,30,800
4,104,Sneha,Delhi,28,450
5,105,Ravi,Hyderabad,25,500
6,106,Kiran,Pune,40,900


In [3]:
print("Duplicate rows:")
display(df_duplicates[df_duplicates.duplicated()])

print("Number of exact duplicate rows:", df_duplicates.duplicated().sum())

Duplicate rows:


,Customer_ID,Name,City,Age,Purchase_Amount
3,102,Priya,Chennai,30,800


Number of exact duplicate rows: 1


In [4]:
all_duplicate_rows = df_duplicates[
    df_duplicates.duplicated(keep=False)
]

display(all_duplicate_rows)

,Customer_ID,Name,City,Age,Purchase_Amount
1,102,Priya,Chennai,30,800
3,102,Priya,Chennai,30,800



## 2. Exact Duplicates

Exact duplicates are rows where every column has the same value.

### When to remove
Remove them when they are confirmed accidental repetitions
and each row should represent a unique record.

### When not to remove
Keep them when repeated rows represent legitimate repeated
events or observations.

### Advantage
Removes redundant records and can prevent double counting.

### Limitation
Blind removal can discard valid repeated events.


In [6]:
print("Repeated Customer_ID values:")

repeated_ids = df_duplicates[
    df_duplicates.duplicated(
        subset=["Customer_ID"],
        keep=False
    )
].sort_values("Customer_ID")

display(repeated_ids)

Repeated Customer_ID values:


,Customer_ID,Name,City,Age,Purchase_Amount
1,102,Priya,Chennai,30,800
3,102,Priya,Chennai,30,800



## 3. Partial Duplicates

Partial duplicates are records that match on selected columns
but differ in one or more other columns.

For example, two rows may share the same customer ID but contain
different addresses or transaction dates.

Partial duplicates require investigation because they may
represent either data-quality problems or legitimate records.

The columns used for matching should be chosen according to
the meaning of the data and its business rules.


In [7]:
df_exact_cleaned = df_duplicates.drop_duplicates()

print("Original shape:", df_duplicates.shape)
print("Shape after exact duplicate removal:", df_exact_cleaned.shape)

display(df_exact_cleaned)

Original shape: (7, 5)
Shape after exact duplicate removal: (6, 5)


,Customer_ID,Name,City,Age,Purchase_Amount
0,101,Ravi,Hyderabad,25,500
1,102,Priya,Chennai,30,800
2,103,Arun,Mumbai,35,650
4,104,Sneha,Delhi,28,450
5,105,Ravi,Hyderabad,25,500
6,106,Kiran,Pune,40,900


In [8]:
print("Keep first occurrence:")
display(df_duplicates.drop_duplicates(subset=["Customer_ID"], keep="first"))

print("Keep last occurrence:")
display(df_duplicates.drop_duplicates(subset=["Customer_ID"], keep="last"))

print("Remove all records belonging to duplicate ID groups:")
display(df_duplicates.drop_duplicates(subset=["Customer_ID"], keep=False))

Keep first occurrence:


,Customer_ID,Name,City,Age,Purchase_Amount
0,101,Ravi,Hyderabad,25,500
1,102,Priya,Chennai,30,800
2,103,Arun,Mumbai,35,650
4,104,Sneha,Delhi,28,450
5,105,Ravi,Hyderabad,25,500
6,106,Kiran,Pune,40,900


Keep last occurrence:


,Customer_ID,Name,City,Age,Purchase_Amount
0,101,Ravi,Hyderabad,25,500
2,103,Arun,Mumbai,35,650
3,102,Priya,Chennai,30,800
4,104,Sneha,Delhi,28,450
5,105,Ravi,Hyderabad,25,500
6,106,Kiran,Pune,40,900


Remove all records belonging to duplicate ID groups:


,Customer_ID,Name,City,Age,Purchase_Amount
0,101,Ravi,Hyderabad,25,500
2,103,Arun,Mumbai,35,650
4,104,Sneha,Delhi,28,450
5,105,Ravi,Hyderabad,25,500
6,106,Kiran,Pune,40,900


In [9]:
duplicate_customer_details = df_duplicates[
    df_duplicates.duplicated(
        subset=["Name", "City", "Age"],
        keep=False
    )
]

display(duplicate_customer_details)

,Customer_ID,Name,City,Age,Purchase_Amount
0,101,Ravi,Hyderabad,25,500
1,102,Priya,Chennai,30,800
3,102,Priya,Chennai,30,800
5,105,Ravi,Hyderabad,25,500


In [10]:
df_selected_cleaned = df_duplicates.drop_duplicates(
    subset=["Name", "City", "Age"],
    keep="first"
)

print("Original row count:", len(df_duplicates))
print("Row count after selected-column deduplication:", len(df_selected_cleaned))

display(df_selected_cleaned)

Original row count: 7
Row count after selected-column deduplication: 5


,Customer_ID,Name,City,Age,Purchase_Amount
0,101,Ravi,Hyderabad,25,500
1,102,Priya,Chennai,30,800
2,103,Arun,Mumbai,35,650
4,104,Sneha,Delhi,28,450
6,106,Kiran,Pune,40,900


In [11]:
customer_updates = pd.DataFrame({
    "Customer_ID": [101, 102, 101, 103, 102],
    "City": ["Hyderabad", "Chennai", "Pune", "Delhi", "Mumbai"],
    "Updated_At": pd.to_datetime([
        "2026-01-01",
        "2026-01-02",
        "2026-02-01",
        "2026-01-03",
        "2026-02-05"
    ])
})

customer_updates = customer_updates.sort_values("Updated_At")

latest_customer_records = customer_updates.drop_duplicates(
    subset=["Customer_ID"],
    keep="last"
)

display(latest_customer_records)

,Customer_ID,City,Updated_At
3,103,Delhi,2026-01-03
2,101,Pune,2026-02-01
4,102,Mumbai,2026-02-05



## 4. Business Rules for Duplicate Removal

Duplicate removal should follow the meaning of the dataset
and the business rules that define a unique record.

Examples:
- Customer master data may require one current record per customer.
- Transaction data may contain multiple legitimate transactions
  from the same customer.
- Medical records may contain repeated tests that must remain
  separate events.
- Time-series data may contain identical measurements at different
  timestamps.

Blindly deleting duplicates can cause loss of legitimate events,
incorrect totals, biased analysis, and incomplete historical records.

Before removing duplicates:
1. Define what makes a record unique.
2. Identify candidate duplicate groups.
3. Inspect conflicting values.
4. Decide which records should be retained.
5. Record the number of rows removed.
6. Validate the cleaned result.


In [12]:
original_rows = len(df_duplicates)
cleaned_rows = len(df_exact_cleaned)
removed_rows = original_rows - cleaned_rows

print("Original rows:", original_rows)
print("Cleaned rows:", cleaned_rows)
print("Rows removed:", removed_rows)
print("Remaining exact duplicates:", df_exact_cleaned.duplicated().sum())

Original rows: 7
Cleaned rows: 6
Rows removed: 1
Remaining exact duplicates: 0



## Conclusion

In this notebook, I learned how to identify exact and partial
duplicates using duplicated() and how to remove duplicates
using drop_duplicates().

I practiced duplicate detection based on selected columns and
explored business rules for retaining the correct records.

I also learned that repeated records are not always errors.
Removing duplicates without understanding the dataset can
discard legitimate transactions, events, or historical information.

Duplicate handling should therefore be based on clear business
rules, careful inspection, and validation.
